In [1]:
# Install the fontTools library
!pip install -q fonttools

from fontTools.ttLib import TTFont
from fontTools.pens.recordingPen import RecordingPen

In [4]:
import math
def extract_full_font(ttc_path, output_filename):
    print(f"Loading {ttc_path}...")
    font = TTFont(ttc_path, fontNumber=0)
    cmap = font.getBestCmap()
    glyph_set = font.getGlyphSet()

    target_chars = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"
    font_dictionary = {}

    print(f"Extracting {len(target_chars)} characters...")

    for char in target_chars:
        char_code = ord(char)
        if char_code not in cmap:
            continue

        glyph_name = cmap[char_code]
        glyph = glyph_set[glyph_name]

        pen = RecordingPen()
        glyph.draw(pen)

        curves = []
        start_pt = (0, 0)
        current_pt = (0, 0)

        for command, args in pen.value:
            if command == 'moveTo':
                current_pt = args[0]
                start_pt = current_pt

            elif command == 'lineTo':
                pt2 = args[0]
                pt1 = ((current_pt[0] + pt2[0]) / 2, (current_pt[1] + pt2[1]) / 2)

                curve = [[round(current_pt[0]), round(current_pt[1])],
                         [round(pt1[0]), round(pt1[1])],
                         [round(pt2[0]), round(pt2[1])]]
                curves.append(curve)
                current_pt = pt2

            elif command == 'qCurveTo':
                pts = list(args)
                endpoint = pts[-1]

                # --- THE FIX IS HERE ---
                # If endpoint is None, the curve wraps back to the start of the loop!
                if endpoint is None:
                    off_curves = pts[:-1]
                    final_dest = start_pt
                else:
                    off_curves = pts[:-1]
                    final_dest = endpoint
                # -----------------------

                prev_pt = current_pt

                for i in range(len(off_curves)):
                    ctrl = off_curves[i]

                    if i < len(off_curves) - 1:
                        next_ctrl = off_curves[i+1]
                        end = ((ctrl[0] + next_ctrl[0]) / 2, (ctrl[1] + next_ctrl[1]) / 2)
                    else:
                        end = final_dest

                    curve = [[round(prev_pt[0]), round(prev_pt[1])],
                             [round(ctrl[0]), round(ctrl[1])],
                             [round(end[0]), round(end[1])]]
                    curves.append(curve)
                    prev_pt = end

                current_pt = final_dest

            elif command == 'closePath':
                if current_pt != start_pt:
                    pt2 = start_pt
                    pt1 = ((current_pt[0] + pt2[0]) / 2, (current_pt[1] + pt2[1]) / 2)
                    curve = [[round(current_pt[0]), round(current_pt[1])],
                             [round(pt1[0]), round(pt1[1])],
                             [round(pt2[0]), round(pt2[1])]]
                    curves.append(curve)
                    current_pt = pt2

        font_dictionary[char] = curves

    print(f"Writing data to {output_filename}...")
    with open(output_filename, 'w') as f:
        f.write("# times_new_roman_dataset.py\n")
        f.write("# Quadratic Bezier Curve Data extracted directly from Times.ttc\n")
        f.write("# Format: { 'Letter': [ [P0, P1, P2], [P0, P1, P2], ... ] }\n\n")
        f.write("font_data = {\n")
        for char, curves in font_dictionary.items():
            f.write(f"    '{char}': {curves},\n")
        f.write("}\n")

    print("Done! You can now download the file.")

# --- RUN THE EXTRACTION ---
extract_full_font('Times.ttc', 'times_new_roman_dataset.py')

Loading Times.ttc...
Extracting 62 characters...
Writing data to times_new_roman_dataset.py...
Done! You can now download the file.
